# 02 - Generation and pairwise files

**Kaggle settings:** Accelerator **GPU T4 x2** (one GPU is used), Internet **On**.
**Add Input -> Notebooks ->** your saved *01 - Training* notebook version (several versions may be attached).

This notebook only reads checkpoints and writes `outputs/eval/generations/*.json` and
`outputs/eval/pairs/*.jsonl`. Decoding is greedy with identical settings for every model.
Section 4 runs each model on 5 prompts and prints the outputs: read them before starting section 5.
Finished files are reused on restart.

## 1. Get the code and install

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

# ---- EDIT THESE ---------------------------------------------------------------------------
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # your GitHub repository
REPO_BRANCH = "main"
CODE_DATASET_DIR = ""   # alternative to GitHub: an uploaded copy, e.g. "/kaggle/input/my-pa2-code/genaiPA2"
# -------------------------------------------------------------------------------------------

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "small-model-alignment"

def find_code_in_inputs():
    """A copy of the repository inside an attached input (an uploaded dataset or an earlier notebook's output)."""
    for marker in sorted(Path("/kaggle/input").glob("**/scripts/judge_pairwise_kaggle.py")):
        return marker.parents[1]
    return None

if REPO_DIR.exists():
    print("repository already present:", REPO_DIR)
else:
    source = Path(CODE_DATASET_DIR) if CODE_DATASET_DIR else None
    if source is None and "YOUR_USERNAME" in REPO_URL:
        source = find_code_in_inputs()
        if source is None:
            raise SystemExit("Set REPO_URL to your GitHub repository, or set CODE_DATASET_DIR, or attach an input "
                             "that contains the code.")
    if source is not None:
        shutil.copytree(source, REPO_DIR, ignore=shutil.ignore_patterns("outputs", ".git", ".venv", "__pycache__"))
        print("copied code from", source)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONUNBUFFERED"] = "1"   # show training / generation logs live in "!python ..." cells
sys.path.insert(0, str(REPO_DIR))
print("working directory:", os.getcwd())

In [ ]:
# Keeps Kaggle's CUDA build of PyTorch: torch is never installed or pinned here.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.56.2,<5" "datasets>=4.7,<5" huggingface_hub "accelerate>=1.4.0" tqdm pyyaml sentencepiece safetensors matplotlib "peft>=0.17.0" "trl==1.14.1" pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!python scripts/check_env.py

## 2. Locate the checkpoints from the training notebook

In [ ]:
from pathlib import Path

def finished(run_name):
    """Directory of a finished run: outputs/<run_name> in this session, else inside any attached input."""
    local = Path("outputs") / run_name
    if (local / "TRAINING_COMPLETE.json").exists():
        return local
    matches = sorted(Path("/kaggle/input").glob(f"**/outputs/{run_name}/TRAINING_COMPLETE.json"))
    return matches[-1].parent if matches else None

for name in ("sft", "dpo_hh", "dpo_ultrafeedback", "persona_lora"):
    print(f"{name:18s} {finished(name) or 'not finished yet'}")

from alignment.io_utils import read_json

def checkpoint(run_name, sub):
    run_dir = finished(run_name)
    return str(run_dir / sub) if run_dir else ""

SFT = checkpoint("sft", "best")
DPO_HH = checkpoint("dpo_hh", "best")
DPO_UF = checkpoint("dpo_ultrafeedback", "best")
PERSONA = checkpoint("persona_lora", "adapter")
# The base model is prompted with the same template the SFT model was trained with.
TEMPLATE = read_json(Path(SFT) / "alignment_meta.json")["template"] if SFT else "alpaca"
print("SFT:", SFT, "\nDPO_HH:", DPO_HH, "\nDPO_UF:", DPO_UF, "\nPERSONA:", PERSONA, "\ntemplate:", TEMPLATE)

## 3. Decoding settings (the same for every model in a comparison)

Greedy decoding, 256 new tokens. If the sanity outputs loop, add an anti-repetition flag for **all** models, for
example `--repetition_penalty 1.1` or `--no_repeat_ngram_size 4`, set `OVERWRITE` below to regenerate, and record
the change in the report.

A saved generation file is reused only for the same checkpoint contents, template and decoding flags. After any
such change the command stops and names the difference instead of reusing old answers; `OVERWRITE = True`
regenerates the files.

In [ ]:
OVERWRITE = False
# Every $NAME used in the "!" lines below is a Python variable (IPython expands a line only if all of them exist).
GEN_FLAGS = "--max_new_tokens 256 --batch_size 16" + (" --overwrite" if OVERWRITE else "")
ALPACA = "eval_sets/alpaca_eval_300_seed42.json"
HH = "eval_sets/hh_rlhf_eval_300_seed42.json"
PERSONA_PROMPTS = "eval_sets/persona_eval_prompts.json"
BASE = "Qwen/Qwen2.5-0.5B"
SMOL = "HuggingFaceTB/SmolLM2-360M-Instruct"

## 4. Sanity check: 5 prompts per model

Writes `*.smoke.json` files only. Check the checkpoint path, the prompt format, that answers stop (EOS) and that
their length is reasonable.

In [ ]:
SMOKE = "--smoke 5"
if SFT:
    !python scripts/generate.py --model_path $BASE --prompts_path $ALPACA --generator qwen2.5-0.5b-base --output_path outputs/eval/generations/alpaca_base.json --template $TEMPLATE $GEN_FLAGS $SMOKE
    !python scripts/generate.py --model_path $SFT --prompts_path $ALPACA --generator qwen2.5-0.5b-sft --output_path outputs/eval/generations/alpaca_sft.json $GEN_FLAGS $SMOKE
    !python scripts/generate.py --model_path $BASE --prompts_path $HH --generator qwen2.5-0.5b-base --output_path outputs/eval/generations/hh_base.json --template $TEMPLATE $GEN_FLAGS $SMOKE
    !python scripts/generate.py --model_path $SFT --prompts_path $HH --generator qwen2.5-0.5b-sft --output_path outputs/eval/generations/hh_sft.json $GEN_FLAGS $SMOKE
if DPO_UF:
    !python scripts/generate.py --model_path $DPO_UF --prompts_path $ALPACA --generator qwen2.5-0.5b-dpo-ultrafeedback --output_path outputs/eval/generations/alpaca_dpo_uf.json $GEN_FLAGS $SMOKE
if DPO_HH:
    !python scripts/generate.py --model_path $DPO_HH --prompts_path $HH --generator qwen2.5-0.5b-dpo-hh --output_path outputs/eval/generations/hh_dpo_hh.json $GEN_FLAGS $SMOKE
if PERSONA:
    !python scripts/generate.py --model_path $SMOL --prompts_path $PERSONA_PROMPTS --generator smollm2-360m-instruct-base --output_path outputs/eval/generations/persona_base.json --template chat_template $GEN_FLAGS $SMOKE
    !python scripts/generate.py --model_path $SMOL --prompts_path $PERSONA_PROMPTS --generator smollm2-360m-instruct-persona-lora --output_path outputs/eval/generations/persona_lora.json --adapter_path $PERSONA --template chat_template $GEN_FLAGS $SMOKE

## 5. Full generation

AlpacaEval subset (300 prompts, seed 42): base, SFT, UltraFeedback-DPO. Helpful/harmless set (300 prompts): base,
SFT, HH-DPO. Persona prompts (40): original SmolLM2-Instruct and the LoRA persona model.

In [ ]:
SMOKE = ""
if SFT:
    !python scripts/generate.py --model_path $BASE --prompts_path $ALPACA --generator qwen2.5-0.5b-base --output_path outputs/eval/generations/alpaca_base.json --template $TEMPLATE $GEN_FLAGS $SMOKE
    !python scripts/generate.py --model_path $SFT --prompts_path $ALPACA --generator qwen2.5-0.5b-sft --output_path outputs/eval/generations/alpaca_sft.json $GEN_FLAGS $SMOKE
    !python scripts/generate.py --model_path $BASE --prompts_path $HH --generator qwen2.5-0.5b-base --output_path outputs/eval/generations/hh_base.json --template $TEMPLATE $GEN_FLAGS $SMOKE
    !python scripts/generate.py --model_path $SFT --prompts_path $HH --generator qwen2.5-0.5b-sft --output_path outputs/eval/generations/hh_sft.json $GEN_FLAGS $SMOKE
if DPO_UF:
    !python scripts/generate.py --model_path $DPO_UF --prompts_path $ALPACA --generator qwen2.5-0.5b-dpo-ultrafeedback --output_path outputs/eval/generations/alpaca_dpo_uf.json $GEN_FLAGS $SMOKE
if DPO_HH:
    !python scripts/generate.py --model_path $DPO_HH --prompts_path $HH --generator qwen2.5-0.5b-dpo-hh --output_path outputs/eval/generations/hh_dpo_hh.json $GEN_FLAGS $SMOKE
if PERSONA:
    !python scripts/generate.py --model_path $SMOL --prompts_path $PERSONA_PROMPTS --generator smollm2-360m-instruct-base --output_path outputs/eval/generations/persona_base.json --template chat_template $GEN_FLAGS $SMOKE
    !python scripts/generate.py --model_path $SMOL --prompts_path $PERSONA_PROMPTS --generator smollm2-360m-instruct-persona-lora --output_path outputs/eval/generations/persona_lora.json --adapter_path $PERSONA --template chat_template $GEN_FLAGS $SMOKE

## 6. Pairwise files (written before any judging)

Response order is randomised with seed 42 (balanced A/B); model identities are stored only in each row's `hidden`
block. The persona pair is built twice: once to judge style faithfulness and once to check that usefulness and
safety held.

Pairing stops with an explanation if one of the two generation files is unfinished or if they were generated with
different template, decoding flags or precision: regenerate the affected file (`OVERWRITE = True`) and rerun.

In [ ]:
G = "outputs/eval/generations"
P = "outputs/eval/pairs"

def have(*names):
    return all((Path(G) / name).exists() for name in names)

if have("alpaca_sft.json", "alpaca_dpo_uf.json"):
    !python scripts/build_pairs.py --baseline_path $G/alpaca_sft.json --candidate_path $G/alpaca_dpo_uf.json --output_path $P/dpo_uf_vs_sft_pairs.jsonl --comparison dpo_uf_vs_sft --profile general
if have("hh_sft.json", "hh_dpo_hh.json"):
    !python scripts/build_pairs.py --baseline_path $G/hh_sft.json --candidate_path $G/hh_dpo_hh.json --output_path $P/dpo_hh_vs_sft_pairs.jsonl --comparison dpo_hh_vs_sft --profile safety
if have("persona_base.json", "persona_lora.json"):
    !python scripts/build_pairs.py --baseline_path $G/persona_base.json --candidate_path $G/persona_lora.json --output_path $P/persona_vs_base_style_pairs.jsonl --comparison persona_vs_base_style --profile persona
    !python scripts/build_pairs.py --baseline_path $G/persona_base.json --candidate_path $G/persona_lora.json --output_path $P/persona_vs_base_quality_pairs.jsonl --comparison persona_vs_base_quality --profile safety
if have("alpaca_base.json", "alpaca_sft.json"):
    # Optional, for your own analysis (not the required comparison): SFT against the untuned base model.
    !python scripts/build_pairs.py --baseline_path $G/alpaca_base.json --candidate_path $G/alpaca_sft.json --output_path $P/sft_vs_base_pairs.jsonl --comparison sft_vs_base --profile general

## 7. Package and save

In [ ]:
!python scripts/summarize_results.py --runs --eval_dir outputs/eval --out_dir outputs/report
!python scripts/package_outputs.py --source outputs/eval outputs/report --zip_path /kaggle/working/eval_generations_and_pairs.zip
!ls -la outputs/eval/generations outputs/eval/pairs

Save the notebook version with its output (**Save Version**; for an interactive run choose Quick Save with
*Always save output*), then attach this notebook to notebook 03.

**Restart:** run all cells again. A finished generation file is reused and an interrupted one continues from its
last completed batch.